# sphinx

Sphinx 9.2.0 turns a tree of reStructuredText (or Markdown, with python/myst_parser) into HTML, text, man pages, LaTeX
and help formats, resolving cross-references across the whole project. These cells build small projects in a
temporary folder: HTML with a toctree and a cross-reference, autodoc + napoleon on a generated module, the text and man
builders, the strict mode that turns a broken reference into an error, and the `sphinx-quickstart` / `sphinx-build`
commands end to end.

In [ ]:
# Example 1: an HTML build with a toctree and a cross-reference, warnings as errors (-W) and nitpicky (-n).
import pathlib, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Demo'\n")
(src / "index.rst").write_text("Demo\n====\n\n.. toctree::\n\n   page\n\nSee :ref:`the-label`.\n")
(src / "page.rst").write_text(".. _the-label:\n\nA page\n======\n\nBody text.\n")
rc = build_main(["-q", "-W", "-n", "-b", "html", str(src), str(src / "_out")])
html = (src / "_out" / "index.html").read_text()
print("rc", rc)
assert rc == 0
assert 'href="page.html#the-label"' in html, html[:400]
assert (src / "_out" / "page.html").exists() and (src / "_out" / "searchindex.js").exists()

In [ ]:
# Example 2: autodoc pulls a module's docstrings in, and napoleon turns a Google-style docstring into fields.
import pathlib, sys, tempfile
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "geo.py").write_text('def area(w, h):\n    """Area of a rectangle.\n\n    Args:\n        w: the width.\n        h: the height.\n\n    Returns:\n        w times h.\n    """\n    return w * h\n')
(src / "conf.py").write_text("import sys\nsys.path.insert(0, %r)\nextensions = ['sphinx.ext.autodoc', 'sphinx.ext.napoleon']\n" % str(src))
(src / "index.rst").write_text("API\n===\n\n.. automodule:: geo\n   :members:\n")
rc = build_main(["-q", "-W", "-b", "html", str(src), str(src / "_out")])
html = (src / "_out" / "index.html").read_text()
print("rc", rc)
assert rc == 0
assert "Area of a rectangle." in html and "Parameters" in html and "the width." in html
assert 'id="geo.area"' in html

In [ ]:
# Example 4: strict mode is what makes Sphinx useful in a gate: a broken reference must FAIL the build, not pass.
import io, pathlib, tempfile
from contextlib import redirect_stderr
from sphinx.cmd.build import build_main

src = pathlib.Path(tempfile.mkdtemp())
(src / "conf.py").write_text("project = 'Broken'\n")
(src / "index.rst").write_text("Broken\n======\n\nSee :ref:`no-such-label` and :doc:`missing`.\n")
err = io.StringIO()
with redirect_stderr(err):
    rc = build_main(["-q", "-W", "--keep-going", "-b", "html", str(src), str(src / "_out")])
print("rc", rc)
print(err.getvalue())
assert rc == 1, rc
assert "undefined label: 'no-such-label'" in err.getvalue()
assert "unknown document: 'missing'" in err.getvalue()